# Totals Feature Engineering

Single source of truth for the 14 totals-specific features used by the totals model.
Loaded by `totals_model.ipynb` and `predict_totals.ipynb` via json+exec (same pattern as `features.ipynb`).

Public surface: `build_totals_features`, `TOTALS_FEATURE_COLS` (14 names), `totals_acc`.

All features are computed on top of the spread model's `g` DataFrame (which already contains
the 35 spread features). Keep this notebook separate from `features.ipynb` — totals features
must never leak into the spread pipeline.

## Parameters

In [ ]:
# Default ON for standalone runs. Consumer notebooks set False before loading.
RUN_TESTS = globals().get('RUN_TESTS', True)


## Imports

In [ ]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')

try:
    import nflreadpy as nfl
except ImportError as _e:
    raise ImportError(f'nflreadpy not found: {_e}') from _e

if RUN_TESTS:
    print('Imports OK.')


## Constants

`TOTALS_FEATURE_COLS` — the 14 totals-specific feature columns in canonical order.
**DO NOT reorder** — column order determines `X_tr` shape which determines pkl identity.

In [ ]:
TOTALS_FEATURE_COLS = [
    # Vegas inputs (3)
    'total_line',
    'home_implied_pts',
    'away_implied_pts',
    # Weather + dome (3)
    'temp_f',
    'wind_mph',
    'is_dome',
    # Rolling scoring (5)
    'home_pts_scored_5g',
    'home_pts_allowed_5g',
    'away_pts_scored_5g',
    'away_pts_allowed_5g',
    'combined_pts_5g',
    # League environment (1)
    'league_avg_total_4wk',
    # Pace + matchup type (2)
    'pace_5g',
    'div_game',
]

if RUN_TESTS:
    assert len(TOTALS_FEATURE_COLS) == 14, f'Expected 14 totals features, got {len(TOTALS_FEATURE_COLS)}'
    assert len(TOTALS_FEATURE_COLS) == len(set(TOTALS_FEATURE_COLS)), 'Duplicate column names'
    print(f'Constants OK: {len(TOTALS_FEATURE_COLS)} totals feature cols')


## Helper — `totals_acc`

Hit-rate against the Vegas total. PUSH games (actual == line) are excluded from
both numerator and denominator (refund convention).

In [ ]:
def totals_acc(preds, total_line, actual_total):
    pred_over   = preds > total_line
    actual_over = actual_total > total_line
    push        = actual_total == total_line
    valid       = ~push
    if not valid.any():
        return 0.0
    return float(((pred_over == actual_over) & valid).sum()) / float(valid.sum())

if RUN_TESTS:
    import numpy as _np
    _p  = _np.array([45.0, 48.0, 41.0, 50.0])
    _tl = _np.array([44.5, 48.0, 43.5, 48.5])
    _a  = _np.array([47.0, 48.0, 40.0, 51.0])
    # game 1: pred OVER, actual OVER -> correct
    # game 2: push -> excluded
    # game 3: pred UNDER, actual UNDER -> correct
    # game 4: pred OVER, actual OVER -> correct
    assert abs(totals_acc(_p, _tl, _a) - 1.0) < 1e-6, 'totals_acc should be 1.0'
    print('totals_acc OK')


## Feature Group — `build_totals_features`

**Inputs:** `g` (spread model games DataFrame with 35 spread features already present),
`sched` (nflreadpy schedule DataFrame), `pbp_full` (raw PBP DataFrame),
`weather_path` (Path to `nfl_weather_*.csv` or None).

**Outputs:** `g` with 14 new columns in `TOTALS_FEATURE_COLS`.

**Note on `is_dome`:** `g['roof']` is ordinal-encoded by the spread pipeline (mc cell 33).
We re-merge the raw roof string from `sched` to correctly detect dome/closed-roof games.
Without this fix, `is_dome` is always 0.

All rolling windows use `shift(1).rolling(N, min_periods=1)` — no within-game leakage.

In [ ]:
def build_totals_features(g, sched, pbp_full, weather_path=None):
    # Defensive: caller must pass sched with the raw `roof` string column
    # (not pre-encoded). is_dome detection depends on this.
    assert 'roof' in sched.columns, \
        "build_totals_features: sched is missing the 'roof' column — is_dome would silently be 0"

    # ── scores + raw roof string ──────────────────────────────────────────────
    # `g` may already carry home_score/away_score (predict_totals path, where
    # game_rows comes from a full_schedule slice) or may not (totals_model
    # path, where g comes from mc cells which drop the scores). Only pull
    # the score columns from sched when g doesn't already have them — pulling
    # both would create _x/_y suffix collisions.
    _aux_cols = ['game_id', 'roof']
    if 'home_score' not in g.columns:
        _aux_cols += ['home_score', 'away_score']
    aux = sched[_aux_cols].rename(columns={'roof': 'roof_str'})
    g = g.merge(aux, on='game_id', how='left')
    g['total_points'] = g['home_score'] + g['away_score']

    # ── implied team totals ───────────────────────────────────────────────────
    g['home_implied_pts'] = (g['total_line'] + g['spread_line']) / 2.0
    g['away_implied_pts'] = (g['total_line'] - g['spread_line']) / 2.0

    # ── dome flag (raw string) ────────────────────────────────────────────────
    g['is_dome'] = g['roof_str'].fillna('outdoors').isin(['dome', 'closed']).astype(int)

    # ── weather (neutralize domes; fill outdoor nulls with mean) ─────────────
    if weather_path is not None and Path(weather_path).exists():
        wx = pd.read_csv(weather_path)[['game_id', 'temp_f', 'wind_mph']]
        g = g.merge(wx, on='game_id', how='left')
    else:
        g['temp_f']   = 60.0   # outdoor league-average fallback
        g['wind_mph'] = 8.0    # outdoor league-average fallback
    dome_mask = g['is_dome'] == 1
    g.loc[dome_mask, 'temp_f']   = 70.0
    g.loc[dome_mask, 'wind_mph'] = 0.0
    g['temp_f']   = g['temp_f'].fillna(g['temp_f'].mean())
    g['wind_mph'] = g['wind_mph'].fillna(g['wind_mph'].mean())

    # ── rolling pts scored / allowed per team (5-game, shift(1)) ─────────────
    hg = sched[['game_id', 'season', 'week', 'home_team', 'home_score', 'away_score']].rename(
        columns={'home_team': 'team', 'home_score': 'pts_scored', 'away_score': 'pts_allowed'})
    ag = sched[['game_id', 'season', 'week', 'away_team', 'away_score', 'home_score']].rename(
        columns={'away_team': 'team', 'away_score': 'pts_scored', 'home_score': 'pts_allowed'})
    long_pts = pd.concat([hg, ag], ignore_index=True).sort_values(['team', 'season', 'week'])
    for col in ['pts_scored', 'pts_allowed']:
        long_pts[f'rolling_{col}_5g'] = (
            long_pts.groupby('team')[col]
            .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean()))
    home_pts = long_pts[['game_id', 'team', 'rolling_pts_scored_5g', 'rolling_pts_allowed_5g']].rename(
        columns={'team': 'home_team',
                 'rolling_pts_scored_5g': 'home_pts_scored_5g',
                 'rolling_pts_allowed_5g': 'home_pts_allowed_5g'})
    away_pts = long_pts[['game_id', 'team', 'rolling_pts_scored_5g', 'rolling_pts_allowed_5g']].rename(
        columns={'team': 'away_team',
                 'rolling_pts_scored_5g': 'away_pts_scored_5g',
                 'rolling_pts_allowed_5g': 'away_pts_allowed_5g'})
    g = g.merge(home_pts, on=['game_id', 'home_team'], how='left')
    g = g.merge(away_pts, on=['game_id', 'away_team'], how='left')
    g['combined_pts_5g'] = (
        g['home_pts_scored_5g'] + g['home_pts_allowed_5g'] +
        g['away_pts_scored_5g'] + g['away_pts_allowed_5g']) / 4.0
    for c in ['home_pts_scored_5g', 'home_pts_allowed_5g',
              'away_pts_scored_5g', 'away_pts_allowed_5g', 'combined_pts_5g']:
        g[c] = g[c].fillna(g[c].mean())

    # ── league scoring environment (rolling 4-week avg) ──────────────────────
    sc = sched[sched['home_score'].notna()].copy()
    sc['game_total'] = sc['home_score'] + sc['away_score']
    weekly_avg = (sc.groupby(['season', 'week'])['game_total']
                  .mean().reset_index()
                  .rename(columns={'game_total': 'week_avg_total'}))
    weekly_avg['league_avg_total_4wk'] = (
        weekly_avg.groupby('season')['week_avg_total']
        .transform(lambda x: x.shift(1).rolling(4, min_periods=1).mean()))
    g = g.merge(weekly_avg[['season', 'week', 'league_avg_total_4wk']],
                on=['season', 'week'], how='left')
    g['league_avg_total_4wk'] = g['league_avg_total_4wk'].fillna(g['league_avg_total_4wk'].mean())

    # ── pace: rolling plays per game (5-game, both teams averaged) ───────────
    plays = (pbp_full[pbp_full['posteam'].notna()]
             .groupby(['game_id', 'posteam']).size().reset_index(name='plays')
             .rename(columns={'posteam': 'team'}))
    week_lkp = sched[['game_id', 'season', 'week']]
    pace_long = (plays.merge(week_lkp, on='game_id', how='left')
                 .sort_values(['team', 'season', 'week']))
    pace_long['rolling_pace_5g'] = (
        pace_long.groupby('team')['plays']
        .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean()))
    for side, col in [('home_team', 'home_pace_5g'), ('away_team', 'away_pace_5g')]:
        lkp = pace_long[['game_id', 'team', 'rolling_pace_5g']].rename(
            columns={'team': side, 'rolling_pace_5g': col})
        g = g.merge(lkp, on=['game_id', side], how='left')
    g['pace_5g'] = (g['home_pace_5g'] + g['away_pace_5g']) / 2.0
    g['pace_5g'] = g['pace_5g'].fillna(g['pace_5g'].mean())

    # ── div_game (already in g from mc pipeline, ensure int) ─────────────────
    g['div_game'] = g['div_game'].fillna(0).astype(int)

    g = g.drop(columns=['roof_str', 'home_pace_5g', 'away_pace_5g'], errors='ignore')
    return g


## Tests — `build_totals_features`

In [ ]:
if RUN_TESTS:
    _rng = np.random.default_rng(42)
    _n = 20
    _ids = [f'2020_{w}_KC_BUF' for w in range(1, _n + 1)]
    _sched_t = pd.DataFrame({
        'game_id': _ids, 'season': 2020, 'week': range(1, _n + 1),
        'home_team': 'KC', 'away_team': 'BUF',
        'home_score': _rng.integers(14, 40, _n).astype(float),
        'away_score': _rng.integers(14, 40, _n).astype(float),
        'roof': 'outdoors', 'div_game': 0,
    })
    _g_t = pd.DataFrame({
        'game_id': _ids, 'season': 2020, 'week': range(1, _n + 1),
        'home_team': 'KC', 'away_team': 'BUF',
        'spread_line': _rng.uniform(-7, 7, _n),
        'total_line':  _rng.uniform(40, 55, _n),
        'roof': _rng.integers(0, 4, _n),  # ordinal-encoded (as mc does it)
        'div_game': 0,
    })
    _pbp_t = pd.DataFrame({
        'game_id': np.repeat(_ids, 60),
        'posteam': np.tile(['KC', 'BUF'], _n * 30),
    })
    _g_out = build_totals_features(_g_t.copy(), _sched_t, _pbp_t, weather_path=None)
    for _col in TOTALS_FEATURE_COLS:
        assert _col in _g_out.columns, f'Missing: {_col}'
        assert _g_out[_col].notna().all(), f'NaN in {_col}'
    _diff = (_g_out['home_implied_pts'] + _g_out['away_implied_pts'] - _g_out['total_line']).abs().max()
    assert _diff < 1e-6, f'Implied total algebra error: {_diff}'
    assert (_g_out['is_dome'] == 0).all(), 'Outdoor game incorrectly flagged as dome'
    # Test dome detection
    _sched_dome = _sched_t.copy(); _sched_dome['roof'] = 'dome'
    _g_dome = build_totals_features(_g_t.copy(), _sched_dome, _pbp_t, weather_path=None)
    assert (_g_dome['is_dome'] == 1).all(), 'Dome game not detected'
    assert (_g_dome['wind_mph'] == 0).all(), 'Dome game should have wind=0'
    assert (_g_dome['temp_f'] == 70).all(), 'Dome game should have temp=70'
    print(f'build_totals_features tests OK | {len(TOTALS_FEATURE_COLS)} features, dome detection OK, algebra OK')


## Cleanup

In [ ]:
if not RUN_TESTS:
    for _tmp in ['_rng', '_n', '_ids', '_sched_t', '_g_t', '_pbp_t', '_g_out',
                 '_col', '_diff', '_sched_dome', '_g_dome']:
        globals().pop(_tmp, None)
